# LangChain Tool Calling

## Setup

- **`langchain`**: For creating AI agents and tools
- **`langchain.chat_models`**: For accessing language models
- **`langchain.agents`**: For creating and managing AI agents

---

## Installing required libraries

In [ ]:
%pip install langchain==0.3.23 | tail -n 1 
%pip install langchain-ibm==0.3.10 | tail -n 1 
%pip install langchain-community==0.3.16 | tail -n 1 
%pip install wikipedia==1.4.0 | tail -n 1
%pip install openai==1.77.0 | tail -n 1
%pip install langchain-openai==0.3.16 | tail -n 1

## Import the required libraries

In [ ]:
from langchain_ibm import ChatWatsonx
from langchain.agents import AgentType
import re

## Loading the LLM: Choosing the right language model

In this example, IBM’s `ChatWatsonx`will be used to load a language model (LLM) for interacting with tools. IBM’s models, like Granite 3.2 and Granite 3.3, are highly versatile and excel at advanced reasoning tasks.

That said, other providers offer LLMs with different strengths:

- **OpenAI (GPT-4/GPT-3.5)**: Best for versatility and advanced reasoning.
- **Facebook (Meta, LLaMA)**: Open-access, highly customizable for specialized use cases.
- **IBM watsonx Granite**: Ideal for enterprise applications with seamless integration.
- **Anthropic (Claude)**: Focused on safety, reliability, and ethical AI.
- **Cohere**: Affordable and efficient for lightweight, task-specific models.

---

- It offers a simple API for quick setup.
- It supports advanced configurations like:
  - **`temperature`**: Adjusting randomness of responses.
  - **`max_tokens`**: Limiting the length of responses.
- IBM’s models are widely regarded as state-of-the-art for general-purpose reasoning and conversation.

In [ ]:
llm = ChatWatsonx(
    model_id="ibm/granite-4-h-small",
    url="https://us-south.ml.cloud.ibm.com",
    project_id="skills-network",
)

Let's generate a simple response:

In [ ]:
response = llm.invoke("What is tool calling in langchain?")
print("\nResponse Content: ", response.content)

## Function

In AI, a **tool** will call a basic **function** or capability that can be called on to perform a specific task. Think of it like a single item in a toolbox: just like a hammer, screwdriver, or wrench, an AI toolbox is full of specific functions designed to solve problems or get things done.

When building tools for tool calling, there are a few key principles to keep in mind:

1. **Clear purpose**: Make sure the tool has a well-defined job.
2. **Standardized input**: The tool should accept input in a predictable, structured format so it’s easy to use.
3. **Consistent output**: Always return results in a format that’s easy to process or integrate with other systems.
4. **Comprehensive documentation**: Your tool should include clear, simple documentation that explains what it does, how to use it, and any quirks or limitations.

Remember, documentation isn’t just for other developers—it’s also for the language model (LLM) to understand the tool’s purpose and how to use it effectively.

In [ ]:
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input dictionary or extracts numbers from a string.

    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.

    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.

    Example Input (Dictionary):
    {"numbers": [10, 20, 30]}

    Example Input (String):
    "Add the numbers 10, 20, and 30."

    Example Output:
    {"result": 60}
    """
    numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]

    
    result = sum(numbers)
    return {"result": result}

In [ ]:
add_numbers("1 2") 

## Tool
The `Tool` class in LangChain serves as a structured wrapper that converts regular Python functions into agent-compatible tools. Each tool needs three key components:
1. A name that identifies the tool
2. The function that performs the actual operation
3. A description that helps the agent understand when to use the tool

Testing section improvement:

In [ ]:
from langchain.agents import Tool
add_tool=Tool(
        name="AddTool",
        func=add_numbers,
        description="Adds a list of numbers and returns the result.")

print("tool object",add_tool)

Let's see the parameters of the object:

- **`name`** (*str*):
  - A unique identifier for the tool.

  - **Example**: `"AddTool"`

- **`.invoke`** (*Callable*):
  - The function that the tool wraps.
  - **Example**: `add_numbers`

- **`description`** (*str*):
  - A concise explanation of what the tool does.
  - **Example**: `"Adds a list of numbers and returns the result."`

In [ ]:
# Tool name
print("Tool Name:")
print(add_tool.name)

# Tool description
print("Tool Description:")
print(add_tool.description)

# Tool function
print("Tool Function:")
print(add_tool.invoke)


In [ ]:
print("Calling Tool Function:")
test_input = "10 20 30 a b" 
print(add_tool.invoke(test_input))  # Example

Testing the tool object ensures:

1. **The tool is correctly set up**:
   - Metadata (`name`, `description`, etc.) is properly defined and aligns with its purpose.
   - The function and schema (if applicable) are correctly configured.

2. **The wrapped function behaves as expected**:
   - The function performs the intended task correctly.
   - It handles edge cases and invalid inputs gracefully.

3. **The tool integrates smoothly with agents**:
   - The tool's output aligns with what the agent expects.
   - There are no compatibility issues when the agent calls the tool.

### `@tool` operator

`@tool` opertor makes tools out of functions. See below:

In [ ]:
from langchain_core.tools import tool
import re

@tool
def add_numbers(inputs:str) -> dict:
    """
    Adds a list of numbers provided in the input string.
    Parameters:
    - inputs (str): 
    string, it should contain numbers that can be extracted and summed.
    Returns:
    - dict: A dictionary with a single key "result" containing the sum of the numbers.
    Example Input:
    "Add the numbers 10, 20, and 30."
    Example Output:
    {"result": 60}
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    # numbers = [int(x) for x in inputs.replace(",", "").split() if x.isdigit()]
    
    result = sum(numbers)
    return {"result": result}

In [ ]:
print("Name: \n", add_numbers.name)
print("Description: \n", add_numbers.description) 
print("Args: \n", add_numbers.args) 


In [ ]:
test_input = "what is the sum between 10, 20 and 30 " 
print(add_numbers.invoke(test_input))  # Example

### Use @tool-StructuredTool

The @tool decorator creates a StructuredTool with schema information extracted from function signatures and docstrings as show here. This helps LLMs better understand what inputs the tool expects and how to use it properly. While both approaches work, @tool is generally preferred for modern LangChain applications, especially with LangGraph and function-calling models.

In [ ]:
# Comparing the two approaches
print("Tool Constructor Approach:")

print(f"Has Schema: {hasattr(add_tool, 'args_schema')}")
print("\n")

print("@tool Decorator Approach:")


print(f"Has Schema: {hasattr(add_numbers, 'args_schema')}")
print(f"Args Schema Info: {add_numbers.args}")

In this example, the tool has two inputs: a string containing the numbers to add, and a second boolean input that determines whether to sum the absolute values of those numbers.

In [ ]:
from typing import List

@tool
def add_numbers_with_options(numbers: List[float], absolute: bool = False) -> float:
    """
    Adds a list of numbers provided as input.

    Parameters:
    - numbers (List[float]): A list of numbers to be summed.
    - absolute (bool): If True, use the absolute values of the numbers before summing.

    Returns:
    - float: The total sum of the numbers.
    """
    if absolute:
        numbers = [abs(n) for n in numbers]
    return sum(numbers)

Let's compare the arguments for add_numbers_with_options and add_numbers. Both are structured tools. They both include the inputs field, which is a string input. However, add_numbers_with_options has an additional key-value pair: absolute, a boolean field with a default value of False. This means add_numbers_with_options supports optional behavior—taking the absolute value of the numbers—while add_numbers only handles basic numeric extraction and summation

In [ ]:
print(f"Args Schema Info: {add_numbers_with_options.args}")
print(f"Args Schema Info: {add_numbers.args}")

In [ ]:
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":False}))
print(add_numbers_with_options.invoke({"numbers":[-1.1,-2.1,-3.0],"absolute":True}))

## Improved tool return types with Python typing

The function `sum_numbers_with_complex_output` returns a more flexible output format. It returns a dictionary containing a float value when numbers are successfully summed, or a descriptive error message as a string if no numbers are found or an issue occurs during processing.

In [ ]:
from typing import Dict, Union

@tool
def sum_numbers_with_complex_output(inputs: str) -> Dict[str, Union[float, str]]:
    """
    Extracts and sums all integers and decimal numbers from the input string.

    Parameters:
    - inputs (str): A string that may contain numeric values.

    Returns:
    - dict: A dictionary with the key "result". If numbers are found, the value is their sum (float). 
            If no numbers are found or an error occurs, the value is a corresponding message (str).

    Example Input:
    "Add 10, 20.5, and -3."

    Example Output:
    {"result": 27.5}
    """
    matches = re.findall(r'-?\d+(?:\.\d+)?', inputs)
    if not matches:
        return {"result": "No numbers found in input."}
    try:
        numbers = [float(num) for num in matches]
        total = sum(numbers)
        return {"result": total}
    except Exception as e:
        return {"result": f"Error during summation: {str(e)}"}

The function `sum_numbers_from_text` returns a straightforward output format. It extracts all integer values from the input string, sums them, and returns the total as a float. This function assumes that at least one valid number is present in the input and does not handle cases where no numbers are found or where an error might occur.

In [ ]:
@tool
def sum_numbers_from_text(inputs: str) -> float:
    """
    Adds a list of numbers provided in the input string.
    
    Args:
        text: A string containing numbers that should be extracted and summed.
        
    Returns:
        The sum of all numbers found in the input.
    """
    # Use regular expressions to extract all numbers from the input
    numbers = [int(num) for num in re.findall(r'\d+', inputs)]
    result = sum(numbers)
    return result

### `initialize_agent`

#### **Relationship between agent and LLM**
- The **agent** acts as the decision-maker, figuring out which tools to use and when.
- The **LLM** is the reasoning engine. It:
  - Interprets the user's input.
  - Helps the agent make decisions.
  - Generates a response based on the output of the tools.

Think of the agent as the manager assigning tasks and the LLM as the brain solving problems or delegating work.

---

#### **Key parameters of `initialize_agent`**

1. **`tools`**- see above 

2.  **`llm`** see above 

3. **`agent`**:
   - Specifies the reasoning framework for the agent.
   - `"zero-shot-react-description"` enables:
     - **Zero-shot reasoning**: The agent can solve tasks it hasn't seen before by thinking through the problem step by step.
     - **React framework**: A logical loop of:
       - **Reason** → Think about the task.
       - **Act** → Use a tool to perform an action.
       - **Observe** → Check the tool's output.
       - **Plan** → Decide what to do next.

4. **`verbose`**:
   - If `True`, it prints detailed logs of the agent’s thought process.
   - Useful for debugging or understanding how the agent makes decisions.

In [ ]:
from langchain.agents import initialize_agent

agent = initialize_agent([add_tool], llm, agent="zero-shot-react-description", verbose=True, handle_parsing_errors=True)

>

In [ ]:
# Use the agent
response =agent.run("In 2023, the US GDP was approximately $27.72 trillion, while Canada's was around $2.14 trillion and Mexico's was about $1.79 trillion what is the total.")

In [ ]:
response

In [ ]:
agent.invoke({"input": "Add 10, 20, two and 30"})

The agent was asked to add the numbers 10, 20, "two," and 30. The agent first noticed that one of the inputs was the word "two" instead of a number, so it converted "two" to its numeric form, which is 2. After preparing the list of numbers (10, 20, 2, and 30), the agent decided to use the `AddTool` to perform the addition. It passed the numbers to the tool, which calculated the sum and returned the result as 62. Finally, the agent provided the answer: **62**.

#### **Structured chat zero shot react-description**

When selecting an agent in LangChain, two factors matter: the agent type and the tool format, especially the tool’s return type. Agents like zero-shot-react-description expect tools to take and return plain strings, which works well with manually defined Tool(...) wrappers. 

In contrast, structured agents like `structured-chat-zero-shot-react-description` or `openai-functions` are built to handle structured inputs and outputs via the @tool decorator. If a tool returns a dictionary but the agent expects a string, it can cause key errors or parsing failures.

In [ ]:
agent_2 = initialize_agent([sum_numbers_from_text], llm, agent="structured-chat-zero-shot-react-description", verbose=True, handle_parsing_errors=True)
response = agent_2.invoke({"input": "Add 10, 20 and 30"})
print(response)

In [ ]:
from langchain_openai import ChatOpenAI

llm_ai = ChatOpenAI(model="gpt-4.1-nano")

In [ ]:
agent_3 = initialize_agent([sum_numbers_with_complex_output], llm_ai, agent="openai-functions", verbose=True, handle_parsing_errors=True)
response = agent_3.invoke({"input": "Add 10, 20 and 30"})
print(response)

In [ ]:
agent_2 = initialize_agent(
    [add_numbers_with_options],
    llm,
    agent="structured-chat-zero-shot-react-description",
    verbose=True
)

In [ ]:
response = agent_2.invoke({
    "input": "Add -10, -20, and -30 using absolute values."
})
print(response)

Let's try with OpenAI to see if it runs with multiple inputs.

In [ ]:
agent_openai = initialize_agent(
    [add_numbers_with_options],
    llm_ai,
    agent="openai-functions",
    verbose=True
)

In [ ]:
response = agent_openai.invoke({
    "input": "Add -10, -20, and -30 using absolute values."
})
print(response)

In [ ]:
%pip install langgraph==0.6.1 | tail -n 1

In [ ]:
from langgraph.prebuilt import create_react_agent

agent_exec = create_react_agent(model=llm_ai, tools=[sum_numbers_from_text])
msgs = agent_exec.invoke({"messages": [("human", "Add the numbers -10, -20, -30")]})

In [ ]:
print(msgs["messages"][-1].content)

## Orchestrating multiple tools with an agent: Mathematical toolkit

### Subtraction tool
The subtraction tool is designed to take a list of numbers and return the result of subtracting all subsequent numbers from the first number. This tool is particularly useful for handling queries involving differences, such as "What is 100 minus 20 and then minus 10?".

In [ ]:
@tool
def subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string, negates the first number, and successively subtracts 
    the remaining numbers in the list.

    This function is designed to handle input in string format, where numbers are separated 
    by spaces, commas, or other delimiters. It parses the string, extracts valid numeric values, 
    and performs a step-by-step subtraction operation starting with the first number negated.

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string may include spaces, commas, or 
      other delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Input:
    "100, 20, 10"

    Example Output:
    {"result": -130}

    Notes:
    - Non-numeric characters in the input are ignored.
    - If the input string contains only one valid number, the result will be that number negated.
    - Handles a variety of delimiters (e.g., spaces, commas) but does not validate input formats 
      beyond extracting numeric values.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number negated
    result = -1 * numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

In [ ]:
print("Name: \n", subtract_numbers.name)
print("Description: \n", subtract_numbers.description) 
print("Args: \n", subtract_numbers.args) 

Let's use it directly by calling the function:

In [ ]:
print("Calling Tool Function:")
test_input = "10 20 30 and four a b" 
print(subtract_numbers.invoke(test_input))  # Example

In [ ]:
# Multiplication Tool
@tool
def multiply_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates their product.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the product of the numbers.

    Example Input:
    "2, 3, 4"

    Example Output:
    {"result": 24}

    Notes:
    - If no numbers are found, the result defaults to 1 (neutral element for multiplication).
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]
    print(numbers)

    # If no numbers are found, return 1
    if not numbers:
        return {"result": 1}

    # Calculate the product of the numbers
    result = 1
    for num in numbers:
        result *= num
        print(num)

    return {"result": result}

In [ ]:
# Division Tool
@tool
def divide_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and calculates the result of dividing the first number 
    by the subsequent numbers in sequence.

    Parameters:
    - inputs (str): A string containing numbers separated by spaces, commas, or other delimiters.

    Returns:
    - dict: A dictionary with the key "result" containing the quotient.

    Example Input:
    "100, 5, 2"

    Example Output:
    {"result": 10.0}

    Notes:
    - If no numbers are found, the result defaults to 0.
    - Division by zero will raise an error.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]


    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Calculate the result of dividing the first number by subsequent numbers
    result = numbers[0]
    for num in numbers[1:]:
        result /= num

    return {"result": result}

In [ ]:
# Testing multiply_tool
multiply_test_input = "2, 3, and four "
multiply_result = multiply_numbers.invoke(multiply_test_input)
print("--- Testing MultiplyTool ---")
print(f"Input: {multiply_test_input}")
print(f"Output: {multiply_result}")

In [ ]:
# Testing divide_tool
divide_test_input = "100, 5, two"
divide_result = divide_numbers.invoke(divide_test_input)
print("--- Testing DivideTool ---")
print(f"Input: {divide_test_input}")
print(f"Output: {divide_result}")

## Building the agent

Let's create the agent object. first, combine all tools into a single list:

In [ ]:
tools = [add_numbers,subtract_numbers, multiply_numbers, divide_numbers]
tools

In [ ]:
from langgraph.prebuilt import create_react_agent

# Create the agent with all tools
math_agent = create_react_agent(
    model=llm_ai,
    tools=tools,
    # Optional: Add a system message to guide the agent's behavior
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)

In [ ]:
response = math_agent.invoke({
    "messages": [("human", "What is 25 divided by 4?")]
})

# Get the final answer
final_answer = response["messages"][-1].content
print(final_answer)

In [ ]:
response_2 = math_agent.invoke({
    "messages": [("human", "Subtract 100, 20, and 10.")]
})

# Get the final answer
final_answer_2 = response_2["messages"][-2].content
print(final_answer_2)

```The Confusion``` 

The agent expects the function to work like normal math 100 - 20 - 10 = 70). When the agent tries to fix this by breaking the problem into smaller steps, it still gets unexpected answers because the calculator keeps using its special rules.

```Getting Stuck```

 The agent keeps trying the same approach repeatedly, not realizing why it isn't working. Eventually, it runs out of time without solving the problem.

In [ ]:
print("\n--- Testing MultiplyTool ---")
response = math_agent.invoke({
    "messages": [("human", "Multiply 2, 3, and four.")]
})
print("Agent Response:", response["messages"][-1].content)

print("\n--- Testing DivideTool ---")
response = math_agent.invoke({
    "messages": [("human", "Divide 100 by 5 and then by 2.")]
})
print("Agent Response:", response["messages"][-1].content)

Now lets change the `SubtractTool` so it subtracts the numbers directly (without negating the first number). This aligns the tool’s behavior with standard arithmetic and the agent’s expectations.

In [ ]:
@tool
def new_subtract_numbers(inputs: str) -> dict:
    """
    Extracts numbers from a string and performs subtraction sequentially, starting with the first number.

    This function is designed to handle input in string format, where numbers may be separated by spaces, 
    commas, or other delimiters. It parses the input string, extracts numeric values, and calculates 
    the result by subtracting each subsequent number from the first. inputs[0]-inputs[1]-inputs[2]

    Parameters:
    - inputs (str): 
      A string containing numbers to subtract. The string can include spaces, commas, or other 
      delimiters between the numbers.

    Returns:
    - dict: 
      A dictionary containing the key "result" with the calculated difference as its value. 
      If no valid numbers are found in the input string, the result defaults to 0.

    Example Usage:
    - Input: "100, 20, 10"
    - Output: {"result": 70}

    Limitations:
    - The function does not handle cases where numbers are formatted with decimals or other non-integer representations.
    """
    # Extract numbers from the string
    numbers = [int(num) for num in inputs.replace(",", "").split() if num.isdigit()]

    # If no numbers are found, return 0
    if not numbers:
        return {"result": 0}

    # Start with the first number
    result = numbers[0]

    # Subtract all subsequent numbers
    for num in numbers[1:]:
        result -= num

    return {"result": result}

## Note: Tool naming when demonstrating different approaches

In this lab, two different ways to create the same mathematical tool (addition) are intentionally shown:

1. Using the `Tool()` constructor approach (`add_tool`)
2. Using the `@tool` decorator approach (`add_numbers`)

When building real agents, duplicate tools with similar functions will confuse the LLM, as it won't know which one to choose. Always use unique tools with clearly differentiated purposes in production code.

Next, create a new agent, ensuring it incorporates the updated subtraction tool.

In [ ]:
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers]
# Create the agent with all tools
math_agent_new = create_react_agent(
    model=llm,
    tools=tools_updated,
    # Optional: Add a system message to guide the agent's behavior
    prompt="You are a helpful mathematical assistant that can perform various operations. Use the tools precisely and explain your reasoning clearly."
)
print("agent",math_agent_new)

In [ ]:
# Test Cases
test_cases = [
    {
        "query": "Subtract 100, 20, and 10.",
        "expected": {"result": 70},
        "description": "Testing subtraction tool with sequential subtraction."
    },
    {
        "query": "Multiply 2, 3, and 4.",
        "expected": {"result": 24},
        "description": "Testing multiplication tool for a list of numbers."
    },
    {
        "query": "Divide 100 by 5 and then by 2.",
        "expected": {"result": 10.0},
        "description": "Testing division tool with sequential division."
    },
    {
        "query": "Subtract 50 from 20.",
        "expected": {"result": -30},
        "description": "Testing subtraction tool with negative results."
    }

]

In [ ]:
correct_tasks = []
# Corrected test execution
for index, test in enumerate(test_cases, start=1):
    query = test["query"]
    expected_result = test["expected"]["result"]  # Extract just the value
    
    print(f"\n--- Test Case {index}: {test['description']} ---")
    print(f"Query: {query}")
    
    # Properly format the input
    response = math_agent_new.invoke({"messages": [("human", query)]})
    
    # Find the tool message in the response
    tool_message = None
    for msg in response["messages"]:
        if hasattr(msg, 'name') and msg.name in ['add_numbers', 'new_subtract_numbers', 'multiply_numbers', 'divide_numbers']:
            tool_message = msg
            break
    
    if tool_message:
        # Parse the tool result from its content
        import json
        tool_result = json.loads(tool_message.content)["result"]
        print(f"Tool Result: {tool_result}")
        print(f"Expected Result: {expected_result}")
        
        if tool_result == expected_result:
            print(f"✅ Test Passed: {test['description']}")
            correct_tasks.append(test["description"])
        else:
            print(f"❌ Test Failed: {test['description']}")
    else:
        print("❌ No tool was called by the agent")

print("\nCorrectly passed tests:", correct_tasks)

The current functions would benefit from enhanced error handling and input validation. The add_numbers, subtract_numbers, multiply_numbers, and divide_numbers functions should be updated to handle decimal numbers using float conversion, validate inputs more strictly, and provide clear error messages for edge cases. For example, divide_numbers should explicitly check for division by zero, and all functions should gracefully handle non-numeric inputs like "two" or "hundred". The test cases should be expanded beyond basic operations to include edge cases like divided by zero, empty inputs, and mixed numeric/text inputs (e.g., "divide one hundred by 5"). Also consider adding tests for decimal numbers (e.g., "multiply 3.5 by 2") and sequential operations (e.g., "multiply 10 by 2 then add 5"). This comprehensive testing approach ensures the agent can handle a wide range of real-world mathematical queries.

## **Exploring LangChain's built-in tools**

---

#### **Why use built-in tools?**
- **Reliability**: Tested and maintained by the LangChain community.
- **Time-saving**: No need to reinvent the wheel for common tasks.
- **Integration**: Designed to work seamlessly with LangChain agents.

---

#### **Popular built-in tools**
Here are some widely used tools from `langchain_community.tools`:

| Tool Name               | Description                                                                 |
|-------------------------|-----------------------------------------------------------------------------|
| `WikipediaQueryRun`     | Search Wikipedia for factual information.                                   |
| `GoogleSearchRun`       | Perform web searches using Google’s API.                                    |
| `PythonREPLTool`        | Execute Python code in a safe environment.                                  |
| `OpenWeatherMapQueryRun`| Fetch real-time weather data.                                               |
| `YouTubeSearchTool`     | Search for YouTube videos.                                                  |

---

#### **Example: Using the Wikipedia tool**
Let’s enhance the math agent with Wikipedia access to answer questions requiring factual context.

In [ ]:
from langchain_community.utilities import WikipediaAPIWrapper

# Create a Wikipedia tool using the @tool decorator
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for factual information about a topic.
    
    Parameters:
    - query (str): The topic or question to search for on Wikipedia
    
    Returns:
    - str: A summary of relevant information from Wikipedia
    """
    wiki = WikipediaAPIWrapper()
    return wiki.run(query)

In [ ]:
search_wikipedia.invoke("What is tool calling?")

- Custom math tools (`add_numbers`, `new_subtract_numbers`, etc.) for arithmetic operations.  
- A built-in tool (`wiki_tool`, e.g., for Wikipedia searches) for additional functionality.  

By combining these tools, the agent can handle **both mathematical calculations** (e.g., addition, subtraction) and **informational queries** (e.g., fetching facts from Wikipedia), depending on the user’s request.

In [ ]:
# Update your tools list to include the Wikipedia tool
tools_updated = [add_numbers, new_subtract_numbers, multiply_numbers, divide_numbers, search_wikipedia]

# Create the agent with all tools including Wikipedia
math_agent_updated = create_react_agent(
    model=llm_ai,
    tools=tools_updated,
    prompt="You are a helpful assistant that can perform various mathematical operations and look up information. Use the tools precisely and explain your reasoning clearly."
)

1. **Online searching** (using `search_wikipedia` or another built-in tool) to fetch real-world data.  
2. **Mathematical computation** (using `multiply_numbers`) to process the retrieved data.

In [ ]:
query = "What is the population of Canada? Multiply it by 0.75"

response = math_agent_updated.invoke({"messages": [("human", query)]})

print("\nMessage sequence:")
for i, msg in enumerate(response["messages"]):
    print(f"\n--- Message {i+1} ---")
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content: {msg.content}")
    if hasattr(msg, 'name'):
        print(f"Name: {msg.name}")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {msg.tool_calls}")

## **Exercise: Create a power tool to calculate exponents**

#### **Objective**

---

#### **Step 1: Create the power tool**

1. **Define the Tool Function**:
   - Create a Python function named `calculate_power` that takes a string as input. The string will contain two numbers: the base (\( x \)) and the exponent (\( y \)).
   - The function should extract the numbers, calculate \( x^y \), and return the result as a dictionary with the key `"result"`.

In [ ]:
def calculate_power(input_text: str) -> dict:
    """Calculates the power of a number (x^y)."""
    match = re.search(r"(\d+(?:\.\d+)?)\s*\^+\s*(\d+(?:\.\d+)?)", input_text)
    if match:
        base, exponent = float(match.group(1)), float(match.group(2))
        return {"result": base ** exponent}

    match = re.search(r"(\d+(?:\.\d+)?)\s*(?:to\s+the\s+power\s+of)\s*(\d+(?:\.\d+)?)", input_text, re.IGNORECASE)
    if match:
        base, exponent = float(match.group(1)), float(match.group(2))
        return {"result": base ** exponent}

    try:
        numbers = [float(n) for n in input_text.replace(",", " ").split()]
        if len(numbers) != 2:
            return {"result": "Invalid input. Provide exactly two numbers."}
        base, exponent = numbers
        return {"result": base ** exponent}
    except ValueError:
        return {"result": "Invalid input format."}

2. **Create the tool object**:
   - Use the `Tool` class from LangChain to create a tool object for the `calculate_power` function.
   - Provide a name, description, and the function to the tool.

In [ ]:
power_tool = Tool(
    name="PowerTool",
    func=calculate_power,
    description="Calculates the power of a number (x^y). Input should be two numbers: base and exponent."
)

#### **Step 2: Create an agent with the power tool**

1. **Set up the agent**:
   - Use the `initialize_agent` function from LangChain to create an agent.
   - Include the `power_tool` in the list of tools provided to the agent.
   - Specify the agent type (e.g., `zero-shot-react-description`).

In [ ]:
tools = [power_tool]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

#### **Step 3: Test the agent**

1. **Test the Agent Using the `run` Function**:
   - Use the `run` function of the agent to test its ability to calculate powers.
   - Pass natural language queries to the agent and observe its responses.

In [ ]:
agent.run("Calculate 5 to the power of 2.")